In [ ]:
import numpy as np
import pandas as pd
import requests
import time
import json
import os
import re
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import kagglehub
from kagglehub import KaggleDatasetAdapter
import warnings

warnings.filterwarnings("ignore")

top_n = 10
min_rating_count = 50
cache_file = "../data/isbn_cache.json"
isbndb_key = ""

# flip this on if i want to load my ratings from the csv instead
use_csv = False
ratings_file = "../data/books_rated.csv"

Load data

In [25]:
print("loading Books.csv")
books = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "arashnic/book-recommendation-dataset",
    "Books.csv",
    pandas_kwargs={"on_bad_lines": "skip", "encoding": "latin-1"},
)

print("loading Ratings.csv")
ratings = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "arashnic/book-recommendation-dataset",
    "Ratings.csv",
    pandas_kwargs={"encoding": "latin-1"},
)

# skip the unrated rows
ratings = ratings[ratings["Book-Rating"] > 0].copy()

print(f"books: {len(books):,}")
print(f"ratings: {len(ratings):,}")
print(f"users: {ratings['User-ID'].nunique():,}")

loading Books.csv
loading Ratings.csv
books: 271,360
ratings: 433,671
users: 77,805


Clean it up

In [26]:
books = books.rename(columns={
    "Book-Title": "title",
    "Book-Author": "author",
    "ISBN": "isbn",
})

def clean_text(value):
    return str(value).strip() if pd.notna(value) else ""

books["title"] = books["title"].apply(clean_text)
books["author"] = books["author"].apply(clean_text)

# only keep books with enough ratings to mean something
rating_counts = ratings.groupby("ISBN")["Book-Rating"].count().rename("n_ratings")
popular_isbns = rating_counts[rating_counts >= min_rating_count].index

books = books[books["isbn"].isin(popular_isbns)].drop_duplicates("isbn").reset_index(drop=True)
print(f"after popularity filter: {len(books):,}")

# merge duplicate editions into one title

def clean_title(title):
    title = title.lower().strip()
    title = re.sub(r"\s*\([^)]*\)", "", title)
    title = re.sub(r"\s*:.*$", "", title)
    return re.sub(r"\s+", " ", title).strip()

books["title_key"] = books["title"].apply(clean_title)
books["rating_count"] = books["isbn"].map(rating_counts).fillna(0).astype(int)

books = (
    books.sort_values("rating_count", ascending=False)
    .drop_duplicates(subset="title_key")
    .drop(columns=["title_key", "rating_count"])
    .reset_index(drop=True)
)
print(f"after deduplication: {len(books):,}")
books[["isbn", "title", "author"]].head(5)

after popularity filter: 531
after deduplication: 462


,isbn,title,author
0,0316666343,The Lovely Bones: A Novel,Alice Sebold
1,0971880107,Wild Animus,Rich Shapero
2,0385504209,The Da Vinci Code,Dan Brown
3,0312195516,The Red Tent (Bestselling Backlist),Anita Diamant
4,0060928336,Divine Secrets of the Ya-Ya Sisterhood: A Novel,Rebecca Wells


Pull extra details from ISBNdb

In [27]:
wait_between_calls = 1.0

def load_cache():
    if os.path.exists(cache_file):
        with open(cache_file) as f:
            return json.load(f)
    return {}

def save_cache(cache_data):
    os.makedirs(os.path.dirname(cache_file), exist_ok=True)
    with open(cache_file, "w") as f:
        json.dump(cache_data, f, indent=2)

def get_book_data(isbn):
    response = requests.get(
        f"https://api2.isbndb.com/book/{isbn}",
        headers={"Authorization": isbndb_key},
        timeout=10,
    )
    if response.status_code == 404:
        return None
    if response.status_code == 429:
        print("\nhit the rate limit, waiting 5s...")
        time.sleep(5)
        return get_book_data(isbn)
    response.raise_for_status()
    book = response.json().get("book", {})
    return {
        "synopsis": (book.get("synopsis") or "").strip(),
        "subjects": book.get("subjects") or [],
    }

isbn_cache = load_cache()
isbns_to_fetch = [isbn for isbn in books["isbn"] if isbn not in isbn_cache]

print(f"total books: {len(books):,}")
print(f"cached: {len(isbn_cache):,}")
print(f"to fetch: {len(isbns_to_fetch):,}\n")

for isbn in tqdm(isbns_to_fetch, desc="ISBNdb"):
    try:
        book_data = get_book_data(isbn)
        isbn_cache[isbn] = book_data if book_data else {"synopsis": "", "subjects": []}
    except Exception as err:
        print(f"\nwarning: {isbn} failed: {err}")
        isbn_cache[isbn] = {"synopsis": "", "subjects": []}
    time.sleep(wait_between_calls)

save_cache(isbn_cache)
print("cache saved")

books["synopsis"] = books["isbn"].map(lambda isbn: isbn_cache.get(isbn, {}).get("synopsis", ""))
books["subjects"] = books["isbn"].map(lambda isbn: ", ".join(isbn_cache.get(isbn, {}).get("subjects", [])[:8]))

def make_book_text(row):
    parts = [f"{row['title']} by {row['author']}" if row["author"] else row["title"]]
    if row["synopsis"]:
        parts.append(row["synopsis"])
    if row["subjects"]:
        parts.append(f"Genres: {row['subjects']}")
    return " | ".join(parts)

books["text"] = books.apply(make_book_text, axis=1)

synopsis_count = (books["synopsis"] != "").sum()
subject_count = (books["subjects"] != "").sum()
print("\ncoverage:")
print(f"synopsis: {synopsis_count:,} / {len(books):,} ({synopsis_count / len(books) * 100:.1f}%)")
print(f"subjects: {subject_count:,} / {len(books):,} ({subject_count / len(books) * 100:.1f}%)")

total books: 462
cached: 531
to fetch: 0



ISBNdb: 0it [00:00, ?it/s]

cache saved

coverage:
synopsis: 460 / 462 (99.6%)
subjects: 460 / 462 (99.6%)


Make embeddings

In [28]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

print(f"embedding {len(books):,} books...")
book_vectors = embed_model.encode(
    books["text"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print(f"shape: {book_vectors.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding 462 books...


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

shape: (462, 384)


Build the profile

In [35]:
def find_book_by_title(title):
    response = requests.get(
        f"https://api2.isbndb.com/books/{requests.utils.quote(title)}",
        headers={"Authorization": isbndb_key},
        params={"pageSize": 5},
        timeout=10,
    )
    if response.status_code != 200:
        return None
    matches = response.json().get("books", [])
    if not matches:
        return None
    title_lower = title.lower()
    best_match = next(
        (book for book in matches if book.get("title", "").lower() == title_lower),
        matches[0],
    )
    return {
        "title": best_match.get("title", title),
        "authors": ", ".join(best_match.get("authors") or []),
        "synopsis": (best_match.get("synopsis") or "").strip(),
        "subjects": best_match.get("subjects") or [],
    }

csv_lookup = {}

if use_csv:
    csv_ratings = pd.read_csv(ratings_file)

    title_col = next((col for col in csv_ratings.columns if "title" in col.lower()), None)
    rating_col = next((col for col in csv_ratings.columns if "rating" in col.lower()), None)
    author_col = next((col for col in csv_ratings.columns if "author" in col.lower()), None)
    synopsis_col = next((col for col in csv_ratings.columns if "synopsis" in col.lower()), None)
    subject_col = next((col for col in csv_ratings.columns if "subject" in col.lower()), None)

    if not title_col or not rating_col:
        raise ValueError(f"couldn't find title/rating columns in {ratings_file}. columns: {list(csv_ratings.columns)}")

    # blank ratings mean i haven't rated it yet
    csv_ratings = csv_ratings[csv_ratings[rating_col].notna()].copy()

    my_ratings = dict(
        zip(
            csv_ratings[title_col].str.strip(),
            csv_ratings[rating_col].astype(float).round().astype(int).clip(0, 5),
        )
    )

    for idx, row in csv_ratings.iterrows():
        title = str(row[title_col]).strip()
        csv_lookup[title] = {
            "authors": str(row[author_col]).strip() if author_col and pd.notna(row[author_col]) else "",
            "synopsis": str(row[synopsis_col]).strip() if synopsis_col and pd.notna(row[synopsis_col]) else "",
            "subjects": str(row[subject_col]).strip() if subject_col and pd.notna(row[subject_col]) else "",
        }

    print(f"loaded {len(my_ratings)} rated books from {ratings_file}\n")

else:
    my_ratings = {
    "The Great Gatsby": 5,
    "Pride and Prejudice": 5,
    "To Kill a Mockingbird": 5,
    "1984": 5,
    "Moby-Dick": 4,
    "Jane Eyre": 5,
    "Wuthering Heights": 4,
    "The Brothers Karamazov": 5,
    "Anna Karenina": 5,
    "The Sound and the Fury": 4,
    "Invisible Man": 5,

    "Divergent": 2,
    "Verity": 2,
    "It Ends with Us": 2,
    "Twilight": 1,
    "The Hunger Games": 2,
    "Ugly Love": 2,
    "A Court of Thorns and Roses": 1,
    "Fourth Wing": 3,
    "The Seven Husbands of Evelyn Hugo": 3,
    }

def score_to_weight(score):
    if score >= 4:
        return float(score)
    if score == 3:
        return 1.0
    if score == 0:
        return -1.0
    return -0.5

title_lookup = {row["title"].lower(): i for i, row in books.iterrows()}

taste_vectors = []
taste_weights = []
rated_titles = []
raw_scores = []
rated_book_indexes = []

for title, score in my_ratings.items():
    weight = score_to_weight(score)

    # try the kaggle data first
    match_idx = title_lookup.get(title.lower())
    if match_idx is None:
        partial_matches = [i for saved_title, i in title_lookup.items() if title.lower() in saved_title]
        if partial_matches:
            match_idx = partial_matches[0]

    if match_idx is not None:
        taste_vectors.append(book_vectors[match_idx])
        taste_weights.append(weight)
        rated_titles.append(books.loc[match_idx, "title"])
        raw_scores.append(score)
        rated_book_indexes.append(match_idx)
        print(f"  [{score} stars  weight={weight:+.1f}]  {books.loc[match_idx, 'title']}  [dataset]")
        continue

    csv_info = csv_lookup.get(title, {})
    synopsis = csv_info.get("synopsis", "")
    subjects = csv_info.get("subjects", "")
    authors = csv_info.get("authors", "")

    if synopsis or subjects:
        parts = [f"{title} by {authors}" if authors else title]
        if synopsis:
            parts.append(synopsis)
        if subjects:
            parts.append(f"Genres: {subjects}")
        vector = embed_model.encode(
            [" | ".join(parts)],
            convert_to_numpy=True,
            normalize_embeddings=True,
        )[0]
        taste_vectors.append(vector)
        taste_weights.append(weight)
        rated_titles.append(title)
        raw_scores.append(score)
        print(f"  [{score} stars  weight={weight:+.1f}]  '{title}'  [csv]")
        continue

    extra_info = find_book_by_title(title)
    if extra_info:
        parts = [f"{extra_info['title']} by {extra_info['authors']}" if extra_info["authors"] else extra_info["title"]]
        if extra_info["synopsis"]:
            parts.append(extra_info["synopsis"])
        if extra_info["subjects"]:
            parts.append(f"Genres: {', '.join(extra_info['subjects'][:8])}")
        vector = embed_model.encode(
            [" | ".join(parts)],
            convert_to_numpy=True,
            normalize_embeddings=True,
        )[0]
        taste_vectors.append(vector)
        taste_weights.append(weight)
        rated_titles.append(extra_info["title"])
        raw_scores.append(score)
        print(f"  [{score} stars  weight={weight:+.1f}]  '{extra_info['title']}'  [ISBNdb]")
        time.sleep(wait_between_calls)
    else:
        print(f"  [{score} stars  weight={weight:+.1f}]  '{title}' not found anywhere, skipping")

print(f"\n{len(taste_vectors)} of {len(my_ratings)} books contributed to the profile")

  [5 stars  weight=+5.0]  The Great Gatsby  [dataset]
  [5 stars  weight=+5.0]  'Pride and Prejudice (Penguin Classics)'  [ISBNdb]
  [5 stars  weight=+5.0]  To Kill a Mockingbird  [dataset]
  [5 stars  weight=+5.0]  1984  [dataset]
  [4 stars  weight=+4.0]  'Moby-Dick'  [ISBNdb]
  [5 stars  weight=+5.0]  'Jane Eyre'  [ISBNdb]
  [4 stars  weight=+4.0]  'Wuthering Heights'  [ISBNdb]
  [5 stars  weight=+5.0]  'The Brothers Karamazov (Hallow Edition): The Classic Russian Novel of Faith, Doubt, and Redemption by Fyodor Dostoevsky (Christian Classics | Ave Maria Press)'  [ISBNdb]
  [5 stars  weight=+5.0]  'Anna Karenina'  [ISBNdb]
  [4 stars  weight=+4.0]  'The Sound and the Fury'  [ISBNdb]
  [5 stars  weight=+5.0]  'Invisible Man'  [ISBNdb]
  [2 stars  weight=-0.5]  'Divergent'  [ISBNdb]
  [2 stars  weight=-0.5]  'Verity'  [ISBNdb]
  [2 stars  weight=-0.5]  'It Ends with Us'  [ISBNdb]
  [1 stars  weight=-0.5]  'Twilight'  [ISBNdb]
  [2 stars  weight=-0.5]  'The Hunger Games (Hunger Games, B

Rank the recommendations

In [36]:
weight_arr = np.array(taste_weights)
user_profile = np.sum(np.vstack(taste_vectors) * weight_arr[:, np.newaxis], axis=0)
user_profile /= np.linalg.norm(user_profile)

# leave out books i've already rated
keep_mask = np.ones(len(books), dtype=bool)
for idx in rated_book_indexes:
    keep_mask[idx] = False

candidate_vectors = book_vectors[keep_mask]
candidate_books = books[keep_mask].copy().reset_index(drop=True)

scores = cosine_similarity([user_profile], candidate_vectors)[0]
candidate_books["score"] = scores

taste_matrix = np.vstack(taste_vectors)
raw_score_arr = np.array(raw_scores)

def guess_rating(candidate_idx):
    vector = candidate_vectors[candidate_idx]
    similar_to_rated = cosine_similarity([vector], taste_matrix)[0]
    positive = similar_to_rated > 0
    if not positive.any():
        return None
    weights = similar_to_rated[positive]
    ratings_used = raw_score_arr[positive]
    return round(float(np.dot(weights, ratings_used) / weights.sum()), 2)

liked_vectors = np.vstack([vector for vector, weight in zip(taste_vectors, taste_weights) if weight > 1.0])
liked_titles = [title for title, weight in zip(rated_titles, taste_weights) if weight > 1.0]

def explain_pick(candidate_idx):
    vector = candidate_vectors[candidate_idx]
    similar_to_liked = cosine_similarity([vector], liked_vectors)[0]
    top_matches = np.argsort(similar_to_liked)[::-1][:2]
    reasons = [liked_titles[i] for i in top_matches if similar_to_liked[i] > 0.1]
    if reasons:
        short_titles = [title if len(title) <= 35 else title[:32] + "..." for title in reasons]
        return "Close to books liked: " + " & ".join(f'"{title}"' for title in short_titles)
    return "Fits overall taste"

top_picks = candidate_books.sort_values("score", ascending=False).head(top_n).copy()
top_picks["predicted_rating"] = [guess_rating(i) for i in top_picks.index]
top_picks["explanation"] = [explain_pick(i) for i in top_picks.index]
top_picks.index = range(1, len(top_picks) + 1)

results = top_picks[["title", "author", "score", "predicted_rating", "explanation"]].copy()
results.columns = ["Title", "Author", "Similarity", "Predicted Rating", "Explanation"]

pd.set_option("display.max_colwidth", 55)
results

,Title,Author,Similarity,Predicted Rating,Explanation
1,A Virtuous Woman (Oprah's Book Club (Paperback)),Kaye Gibbons,0.607153,3.49,"Close to books liked: ""Anna Karenina"" & ""Jane Eyre"""
2,Beloved (Plume Contemporary Fiction),Toni Morrison,0.600939,3.50,"Close to books liked: ""To Kill a Mockingbird"" & ""An..."
3,The Notebook,Nicholas Sparks,0.592128,3.50,"Close to books liked: ""To Kill a Mockingbird"" & ""An..."
4,The Eyre Affair: A Novel,Jasper Fforde,0.571304,3.49,"Close to books liked: ""Wuthering Heights"" & ""Pride ..."
5,The Sparrow,MARY DORIA RUSSELL,0.569949,3.60,"Close to books liked: ""To Kill a Mockingbird"" & ""An..."
6,The Reader,Bernhard Schlink,0.568601,3.39,"Close to books liked: ""Anna Karenina"" & ""To Kill a ..."
7,Envy,Sandra Brown,0.565114,3.38,"Close to books liked: ""Anna Karenina"" & ""Jane Eyre"""
8,Here on Earth,Alice Hoffman,0.557290,3.42,"Close to books liked: ""Anna Karenina"" & ""To Kill a ..."
9,American Psycho (Vintage Contemporaries),Bret Easton Ellis,0.553170,3.57,"Close to books liked: ""Invisible Man"" & ""Moby-Dick"""
10,Cold Mountain : A Novel,CHARLES FRAZIER,0.552128,3.63,"Close to books liked: ""Invisible Man"" & ""To Kill a ..."


Check a single ISBN

In [37]:
def check_isbn(isbn):
    print(f"querying ISBNdb for {isbn}...")
    response = requests.get(
        f"https://api2.isbndb.com/book/{isbn}",
        headers={"Authorization": isbndb_key},
        timeout=10,
    )
    if response.status_code == 404: # not found
        print("not found on ISBNdb")
        return None
    if response.status_code == 429: # too many requests
        print("rate limited")
        return None
    response.raise_for_status()

    book = response.json().get("book", {})
    title = (book.get("title") or "").strip()
    authors = ", ".join(book.get("authors") or [])
    synopsis = (book.get("synopsis") or "").strip()
    subjects = book.get("subjects") or []

    if not title:
        print("ISBNdb returned no title for this ISBN")
        return None

    print(f"Title: {title}")
    print(f"Author: {authors or 'unknown'}")
    print(f"Subjects: {', '.join(subjects[:5]) if subjects else 'not available'}")

    parts = [f"{title} by {authors}" if authors else title]
    if synopsis:
        parts.append(synopsis)
    if subjects:
        parts.append(f"Genres: {', '.join(subjects[:8])}")

    vector = embed_model.encode(
        [" | ".join(parts)],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )[0]

    similarity = float(cosine_similarity([vector], [user_profile])[0][0])

    similar_to_rated = cosine_similarity([vector], taste_matrix)[0]
    positive = similar_to_rated > 0
    if positive.any():
        weights = similar_to_rated[positive]
        ratings_used = raw_score_arr[positive]
        predicted_rating = round(float(np.dot(weights, ratings_used) / weights.sum()), 2)
    else:
        predicted_rating = None

    similar_to_liked = cosine_similarity([vector], liked_vectors)[0]
    top_matches = np.argsort(similar_to_liked)[::-1][:2]
    reasons = [liked_titles[i] for i in top_matches if similar_to_liked[i] > 0.1]

    print(f"\nSimilarity to profile: {similarity:.4f}")
    print(f"Predicted rating: {predicted_rating} / 5" if predicted_rating else "Predicted rating: n/a")
    if reasons:
        print(f"Close to books liked: {' & '.join(reasons)}")

    return {
        "title": title,
        "authors": authors,
        "similarity": similarity,
        "predicted": predicted_rating,
        "reasons": reasons,
    }

In [38]:
check_isbn("9781250301697")

querying ISBNdb for 9781250301697...
Title: The Silent Patient
Author: Alex Michaelides
Subjects: Fiction, Thrillers, Psychological, Suspense

Similarity to profile: 0.3422
Predicted rating: 3.27 / 5
Close to books liked: Anna Karenina & Jane Eyre


{'title': 'The Silent Patient',
 'authors': 'Alex Michaelides',
 'similarity': 0.34222788732931886,
 'predicted': 3.27,
 'reasons': ['Anna Karenina', 'Jane Eyre']}

In [39]:
check_isbn("9780439023528")

querying ISBNdb for 9780439023528...
Title: The Hunger Games
Author: Suzanne Collins
Subjects: Juvenile Fiction, Action & Adventure, Survival Stories, Performing Arts, Television & Radio

Similarity to profile: 0.3683
Predicted rating: 3.25 / 5
Close to books liked: Anna Karenina & The Sound and the Fury


{'title': 'The Hunger Games',
 'authors': 'Suzanne Collins',
 'similarity': 0.36830615292594593,
 'predicted': 3.25,
 'reasons': ['Anna Karenina', 'The Sound and the Fury']}

In [41]:
check_isbn("9781594138539")

querying ISBNdb for 9781594138539...
Title: Insurgent
Author: Veronica Roth
Subjects: Juvenile Fiction, Family, JUVENILE FICTION, Love & Romance, Fantasy & Magic

Similarity to profile: 0.4117
Predicted rating: 3.23 / 5
Close to books liked: Anna Karenina & The Sound and the Fury


{'title': 'Insurgent',
 'authors': 'Veronica Roth',
 'similarity': 0.41165640719746466,
 'predicted': 3.23,
 'reasons': ['Anna Karenina', 'The Sound and the Fury']}